# Lab Task 4: Text Pre-Processing Techniques
**Course:** DL-3002 Data Mining Lab

This notebook applies noise removal, normalization, n-gram extraction, frequency analysis, word/sentence tokenization, subword tokenization, and a discussion of contextualized tokenization to the news dataset scraped in the previous lab (`output.csv`).

**Note on the dataset:** `output.csv` contains three columns — `headline`, `date`, `source`. There is no separate full-article body column, so the `headline` field is used as the text ("news body") on which all pre-processing steps below are demonstrated.

In [1]:
import re
import pandas as pd
import nltk
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.collocations import BigramCollocationFinder, TrigramCollocationFinder
from nltk.metrics import BigramAssocMeasures, TrigramAssocMeasures
from nltk import FreqDist
from tokenizers import BertWordPieceTokenizer
from huggingface_hub import hf_hub_download

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

pd.set_option('display.max_colwidth', 100)
print("Setup complete.")

Setup complete.

## 1. Noise Removal
- **(a) HTML tags and URLs:** `remove_html_and_urls` strips any HTML markup with BeautifulSoup and removes URLs with a regex.
- **(b) Special characters and punctuation:** `remove_special_characters` keeps only word characters, whitespace, periods, and commas.
- **(c) Case conversion:** `to_lowercase` lowercases everything for consistency.

In [2]:
df = pd.read_csv(r"C:\Users\Hassan Younas\OneDrive\Documents\Batch' 21\Data Mining\Lab\Lab 4\output.csv", encoding='utf-8')
print(df.shape)
print(df.head())

(2600, 3)
                                                                  headline  ...             source
0                          Iran prepares for late supreme leader's funeral  ...  Channel News Asia
1  ‘Booby-trapped package’ explodes in Monaco, injuring Ukrainian oligarch  ...        Global News
2      Ukraine live: ‘Putin ordered army to plan new ways to capture Kyiv’  ...    The Independent
3     Commentary: Small states are the best bet for keeping world at peace  ...  Channel News Asia
4           China slaps 73.5% preliminary tariff on pea starch from Canada  ...        Global News

[5 rows x 3 columns]

In [3]:
def remove_html_and_urls(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    return text

def remove_special_characters(text):
    return re.sub(r'[^\w\s,\.]', '', text)

def to_lowercase(text):
    return text.lower()

df['headline'] = df['headline'].astype(str)
df['clean_headline'] = df['headline'].apply(remove_html_and_urls)
df['clean_headline'] = df['clean_headline'].apply(remove_special_characters)
df['clean_headline'] = df['clean_headline'].apply(to_lowercase)
print(df[['headline', 'clean_headline']].head())

                                                                  headline                                                        clean_headline
0                          Iran prepares for late supreme leader's funeral                        iran prepares for late supreme leaders funeral
1  ‘Booby-trapped package’ explodes in Monaco, injuring Ukrainian oligarch  boobytrapped package explodes in monaco, injuring ukrainian oligarch
2      Ukraine live: ‘Putin ordered army to plan new ways to capture Kyiv’      ukraine live putin ordered army to plan new ways to capture kyiv
3     Commentary: Small states are the best bet for keeping world at peace   commentary small states are the best bet for keeping world at peace
4           China slaps 73.5% preliminary tariff on pea starch from Canada         china slaps 73.5 preliminary tariff on pea starch from canada

## 2. Normalization
### (a) Stemming vs. Lemmatization
The table below compares Porter stemming and WordNet lemmatization on the same tokens.

- **Stemming** (`explodes -> explod`, `package -> packag`) chops suffixes with fixed rules. It is fast and cheap but frequently produces non-words, which hurts readability and can merge unrelated words.
- **Lemmatization** (`explodes -> explodes`, `leaders -> leader`) uses vocabulary and morphological analysis to return a real dictionary form. It is slower but much more interpretable.

For this dataset, headlines are short and will later be inspected as n-grams and frequency tables, so readability matters — **lemmatization** is used for the rest of the pipeline.

In [4]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

sample = df['clean_headline'].iloc[1]
tokens = word_tokenize(sample)
stems = [stemmer.stem(t) for t in tokens]
lemmas = [lemmatizer.lemmatize(t) for t in tokens]
comparison = pd.DataFrame({'token': tokens, 'stem': stems, 'lemma': lemmas})
print(comparison)

          token       stem         lemma
0  boobytrapped  boobytrap  boobytrapped
1       package     packag       package
2      explodes     explod      explodes
3            in         in            in
4        monaco     monaco        monaco
5             ,          ,             ,
6      injuring      injur      injuring
7     ukrainian  ukrainian     ukrainian
8      oligarch   oligarch      oligarch

In [5]:
def lemmatize_text(text):
    toks = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(t) for t in toks)

df['lemmatized_headline'] = df['clean_headline'].apply(lemmatize_text)
print(df[['clean_headline', 'lemmatized_headline']].head())

                                                         clean_headline                                                    lemmatized_headline
0                        iran prepares for late supreme leaders funeral                          iran prepares for late supreme leader funeral
1  boobytrapped package explodes in monaco, injuring ukrainian oligarch  boobytrapped package explodes in monaco , injuring ukrainian oligarch
2      ukraine live putin ordered army to plan new ways to capture kyiv        ukraine live putin ordered army to plan new way to capture kyiv
3   commentary small states are the best bet for keeping world at peace     commentary small state are the best bet for keeping world at peace
4         china slaps 73.5 preliminary tariff on pea starch from canada           china slap 73.5 preliminary tariff on pea starch from canada

### (b) Stop Words Removal
Common English stop words (e.g. *the, is, at, of*) are removed since they carry little topical meaning and would otherwise dominate frequency and n-gram statistics.

In [6]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    toks = word_tokenize(text)
    filtered = [t for t in toks if t not in stop_words]
    return ' '.join(filtered)

df['final_headline'] = df['lemmatized_headline'].apply(remove_stopwords)
print(df[['headline', 'final_headline']].head())

                                                                  headline                                                      final_headline
0                          Iran prepares for late supreme leader's funeral                           iran prepares late supreme leader funeral
1  ‘Booby-trapped package’ explodes in Monaco, injuring Ukrainian oligarch  boobytrapped package explodes monaco , injuring ukrainian oligarch
2      Ukraine live: ‘Putin ordered army to plan new ways to capture Kyiv’           ukraine live putin ordered army plan new way capture kyiv
3     Commentary: Small states are the best bet for keeping world at peace                 commentary small state best bet keeping world peace
4           China slaps 73.5% preliminary tariff on pea starch from Canada                china slap 73.5 preliminary tariff pea starch canada

## 3. Extraction of N-grams
Unigrams, bigrams, and trigrams are generated with NLTK's `word_tokenize`, `nltk.bigrams`, and `nltk.trigrams` over the fully cleaned `final_headline` text. `BigramCollocationFinder` / `TrigramCollocationFinder` with PMI (pointwise mutual information) are then used to surface strongly *associated* word pairs/triples, as opposed to merely frequent ones (a low-frequency filter is applied first so PMI isn't dominated by one-off pairs).

In [7]:
all_tokens = word_tokenize(' '.join(df['final_headline']))
unigrams = all_tokens
bigrams_list = list(nltk.bigrams(all_tokens))
trigrams_list = list(nltk.trigrams(all_tokens))
print("Sample unigrams:", unigrams[:10])
print("Sample bigrams:", bigrams_list[:10])
print("Sample trigrams:", trigrams_list[:10])

bigram_finder = BigramCollocationFinder.from_words(all_tokens)
trigram_finder = TrigramCollocationFinder.from_words(all_tokens)
bigram_finder.apply_freq_filter(3)
trigram_finder.apply_freq_filter(2)

top10_bigrams_pmi = bigram_finder.nbest(BigramAssocMeasures.pmi, 10)
top10_trigrams_pmi = trigram_finder.nbest(TrigramAssocMeasures.pmi, 10)
print("Top 10 bigram collocations (PMI):", top10_bigrams_pmi)
print("Top 10 trigram collocations (PMI):", top10_trigrams_pmi)

Sample unigrams: ['iran', 'prepares', 'late', 'supreme', 'leader', 'funeral', 'boobytrapped', 'package', 'explodes', 'monaco']
Sample bigrams: [('iran', 'prepares'), ('prepares', 'late'), ('late', 'supreme'), ('supreme', 'leader'), ('leader', 'funeral'), ('funeral', 'boobytrapped'), ('boobytrapped', 'package'), ('package', 'explodes'), ('explodes', 'monaco'), ('monaco', ',')]
Sample trigrams: [('iran', 'prepares', 'late'), ('prepares', 'late', 'supreme'), ('late', 'supreme', 'leader'), ('supreme', 'leader', 'funeral'), ('leader', 'funeral', 'boobytrapped'), ('funeral', 'boobytrapped', 'package'), ('boobytrapped', 'package', 'explodes'), ('package', 'explodes', 'monaco'), ('explodes', 'monaco', ','), ('monaco', ',', 'injuring')]
Top 10 bigram collocations (PMI): [('beauty', 'salon'), ('kim', 'jong'), ('negeri', 'sembilan'), ('buckingham', 'palace'), ('daveigh', 'chase'), ('per', 'cent'), ('preston', 'davey'), ('taylor', 'swift'), ('hong', 'kong'), ('giving', 'birth')]
Top 10 trigram col

## 4. Frequency Analysis
`nltk.FreqDist` is used to count occurrences of every unigram, bigram, and trigram across the whole corpus, and the top 10 most common of each are displayed. Unlike the PMI collocations above (which reward strong association even at low counts), this ranks purely by raw frequency — so common news-writing patterns like `('world', 'cup')` or `(',', 'say')` dominate.

In [8]:
unigram_freq = FreqDist(unigrams)
bigram_freq = FreqDist(bigrams_list)
trigram_freq = FreqDist(trigrams_list)

print("Top 10 unigrams by frequency:")
for w, c in unigram_freq.most_common(10):
    print(f"  {w!r}: {c}")

print("Top 10 bigrams by frequency:")
for w, c in bigram_freq.most_common(10):
    print(f"  {w}: {c}")

print("Top 10 trigrams by frequency:")
for w, c in trigram_freq.most_common(10):
    print(f"  {w}: {c}")

Top 10 unigrams by frequency:
  ',': 623
  'say': 273
  'iran': 242
  'trump': 201
  'new': 146
  'u.s.': 124
  'canada': 118
  'war': 115
  '.': 103
  'world': 99
Top 10 bigrams by frequency:
  ('world', 'cup'): 65
  ('iran', 'war'): 55
  ('trump', 'say'): 33
  (',', 'say'): 32
  ('strait', 'hormuz'): 32
  ('oil', 'price'): 28
  ('b.c', '.'): 22
  ('ebola', 'outbreak'): 21
  ('middle', 'east'): 20
  ('peace', 'deal'): 17
Top 10 trigrams by frequency:
  ('amid', 'iran', 'war'): 11
  (',', 'official', 'say'): 7
  (',', 'study', 'find'): 7
  ('oil', 'price', 'fall'): 7
  ('world', 'cup', 'match'): 6
  ('norway', 'crown', 'princess'): 6
  (',', 'expert', 'say'): 6
  ('iran', 'war', ','): 6
  ('king', 'charles', 'iii'): 5
  ('son', 'norway', 'crown'): 5

## 5. Tokenization
Tokenization is explored at two granularities: **word-level** (`word_tokenize`) and **sentence-level** (`sent_tokenize`). Since each row in this dataset is a single standalone headline (no internal sentence boundaries), five headlines are joined into one pseudo-document (separated by periods) purely to demonstrate sentence tokenization meaningfully.

In [9]:
sample_doc = '. '.join(df['headline'].iloc[:5]) + '.'
word_level = word_tokenize(sample_doc)
sentence_level = sent_tokenize(sample_doc)
print("Word-level tokens (first 20):", word_level[:20])
print("Number of word tokens:", len(word_level))
print("Sentence-level tokens:")
for i, s in enumerate(sentence_level, 1):
    print(f"  {i}. {s}")

Word-level tokens (first 20): ['Iran', 'prepares', 'for', 'late', 'supreme', 'leader', "'s", 'funeral', '.', '‘', 'Booby-trapped', 'package', '’', 'explodes', 'in', 'Monaco', ',', 'injuring', 'Ukrainian', 'oligarch']
Number of word tokens: 63
Sentence-level tokens:
  1. Iran prepares for late supreme leader's funeral.
  2. ‘Booby-trapped package’ explodes in Monaco, injuring Ukrainian oligarch.
  3. Ukraine live: ‘Putin ordered army to plan new ways to capture Kyiv’.
  4. Commentary: Small states are the best bet for keeping world at peace.
  5. China slaps 73.5% preliminary tariff on pea starch from Canada.

## 6. Subword Tokenization
BERT's WordPiece tokenizer breaks unknown/rare words into known sub-word pieces (prefixed with `##`) instead of mapping them to a single `[UNK]` token. This is demonstrated below using the `tokenizers` library's `BertWordPieceTokenizer` loaded with the official `bert-base-uncased` vocabulary (downloaded via `huggingface_hub`).

*Environment note:* this machine's local `torch` install is broken (DLL load failure), which blocks importing the full `transformers` package (it unconditionally probes for torch). `tokenizers` + the raw vocab file gives the identical WordPiece tokenization without needing torch, so the subword-tokenization behavior below is real, not simulated.

Notice how `'boobytrapped'` (an unusual compound created after punctuation removal) splits into `['boo', '##by', '-', 'trapped']`-style pieces, and `'preprocessing'`/`'tokenization'` split into recognizable morphemes (`prep`, `##ro`, `##ces`, `##sing`) rather than becoming a single out-of-vocabulary token.

In [10]:
vocab_path = hf_hub_download(repo_id='bert-base-uncased', filename='vocab.txt')
bert_tokenizer = BertWordPieceTokenizer(vocab_path, lowercase=True)

oov_examples = [
    df['headline'].iloc[1],
    df['headline'].iloc[8],
    'preprocessing tokenization deportation booby-trapped',
]
for text in oov_examples:
    enc = bert_tokenizer.encode(text)
    print(f"Text: {text}")
    print(f"  WordPiece tokens: {enc.tokens}")
    print()

Text: ‘Booby-trapped package’ explodes in Monaco, injuring Ukrainian oligarch
  WordPiece tokens: ['[CLS]', '‘', 'boo', '##by', '-', 'trapped', 'package', '’', 'explodes', 'in', 'monaco', ',', 'injuring', 'ukrainian', 'ol', '##iga', '##rch', '[SEP]']

Text: Russia arms civilian gas ship with machine guns in ‘warning to Nato’
  WordPiece tokens: ['[CLS]', 'russia', 'arms', 'civilian', 'gas', 'ship', 'with', 'machine', 'guns', 'in', '‘', 'warning', 'to', 'nato', '’', '[SEP]']

Text: preprocessing tokenization deportation booby-trapped
  WordPiece tokens: ['[CLS]', 'prep', '##ro', '##ces', '##sing', 'token', '##ization', 'deportation', 'boo', '##by', '-', 'trapped', '[SEP]']

## 7. Contextualized Tokenization
WordPiece (task 6) is *static*: a given sub-word always maps to the same vocabulary ID no matter where it appears. **Contextualized tokenization**, as used inside BERT, goes one step further — after the WordPiece IDs are looked up, they pass through many self-attention transformer layers so that each token's final vector representation is a function of its *entire surrounding sentence*. The word **"strike"** would end up with a very different embedding in *"oil strike found off the coast"* versus *"workers plan a strike over pay"*, even though the WordPiece token id is identical in both cases — something a static bag-of-words/n-gram count (tasks 3-4) can never capture, since those treat every occurrence of a word identically regardless of context.

**Would this help here?** Yes. The frequency/collocation analysis above already hints at ambiguity risk in this corpus — words like `'strike'`, `'crown'` (princess vs. sports), or `'war'` recur in different senses across `Channel News Asia`, `Global News`, and `The Independent` headlines. Contextual embeddings would let downstream tasks (topic clustering, near-duplicate headline detection across the three sources, sentiment) disambiguate these senses automatically, instead of relying on raw token/n-gram overlap.

*Not executed here:* producing actual contextual embeddings requires a full forward pass through `BertModel`, which needs a working `torch` install (broken on this machine, see the note in task 6). In a working environment this would simply be:
```python
from transformers import BertTokenizer, BertModel
tok = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
inputs = tok(headline, return_tensors='pt')
last_hidden_state = model(**inputs).last_hidden_state  # one context-dependent vector per token
```